## Lab 3 Solution using OpenAI: praneeth_39

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import display, Markdown
import gradio as gr
import json

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
reader = PdfReader("../twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [ ]:
with open("../twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
print(summary)

In [ ]:
system_prompt = f"""
You are a professional digital twin representing this person.

Answer questions only about the person's career, background, skills, experience,
education, products, services, expertise, and business area.
Use only the information provided below. Do not invent information.

If the question is unrelated to work, politely redirect the user to professional topics.
Never invent information. If the answer is not available in the context, say so.

If a user wants to get in touch, ask for their email address and use the
record_email_tool after they provide it.

Summary:
{summary}

Business and professional information:
{linkedin}
"""

In [ ]:
display(Markdown(system_prompt))

### First LLM Call

In [ ]:
def generate_reply(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(
        model="gpt-5.4-mini",
        messages=messages
    )

    return response.choices[0].message.content

### Second LLM Call to check user's question and the proposed answer


In [ ]:
evaluator_prompt_template = """
    You are a strict content evaluator.

    Return work_related=true for:
    - career, skills, experience, education, or business questions;
    - products, services, or professional expertise;
    - requests to contact the person or business;
    - requests to record an email address;
    - normal greetings such as hi, hello, or hey.

    Return work_related=false only for clearly unrelated topics such as jokes,
    entertainment, politics, or personal topics.

    Return JSON only: 
    {{"work_related": true}}

    User request:
    {user_query}

    Proposed assistant answer:
    {proposed_answer}
"""

In [ ]:
def evaluate_work_related(message, draft_reply):
    evaluator_prompt = evaluator_prompt_template.format(
        user_query=message,
        proposed_answer=draft_reply or ""
    )

    messages = [
        {"role": "system", "content": "Return valid JSON only"},
        {"role": "user", "content": evaluator_prompt}
    ]

    response = openai.chat.completions.create(
        model="gpt-5.4-mini",
        messages=messages,
        response_format={"type": "json_object"}
    )

    result =  json.loads(response.choices[0].message.content)
    return bool(result.get("work_related", False))

### Combine both LLM calls

In [ ]:
def chat(message, history):
    draft_reply = generate_reply(message, history)

    is_work_related = evaluate_work_related(message, draft_reply)

    if is_work_related:
        return draft_reply

    return (
        "I can only help with professional topics such as career, "
        "background, skills, and experience."
    )

In [ ]:
#gr.ChatInterface(chat).launch(inbrowser=True)

### Email Tool Function

In [ ]:
def record_email_tool(emails):
    if isinstance(emails, str):
        emails = [emails] 
    recorded = []

    with open("../emails.txt", "a", encoding="utf-8") as f:
        for email in emails:
            email = email.strip()

            if email and email not in recorded:
                print(f"Recording Email: {email}")
                f.write(email + "\n")
                recorded.append(email)

    return f"Recorded {len(recorded)} email address(es)" 

In [ ]:
#record_email_tool("test@praneeth_39.com")

### json to decribe the tool

In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Record one or more email addresses",
    "parameters": {
        "type": "object",
        "properties": {
            "emails": {
                "type": "array", 
                "items": {"type": "string"}, 
                "description": "The user's email address"
            }
        },
        "required": ["emails"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [{"type": "function", "function": record_email_tool_json}]

tools

In [ ]:
def chatv2(message, history):
    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        messages.append(assistant_message)
        for tool_call in assistant_message.tool_calls:
            arguments = json.loads(tool_call.function.arguments)
            emails = arguments.get("emails", arguments.get("email", []))
            tool_result = record_email_tool(emails)
            messages.append({"role": "tool", "content": tool_result, "tool_call_id": tool_call.id})

        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    draft_reply = response.choices[0].message.content or ""
    is_work_related = evaluate_work_related(message, draft_reply)

    if is_work_related:
        return draft_reply

    return "I can only help with professional or business-related topics."

In [ ]:
gr.ChatInterface(chatv2).launch(inbrowser=True)